# 04 — County GIS Analysis

County-level unemployment, income, and UI exposure mapping.

In [ ]:
import json, pandas as pd
import plotly.express as px
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
county_path = ROOT / "data" / "county_data.json"

if not county_path.exists():
    print("County data not found. Run: python fetch_county_data.py")
else:
    with open(county_path) as f:
        data = json.load(f)
    rows = []
    for c in data["counties"]:
        rates = c.get("unemployment_by_year", {})
        rows.append({
            "fips": c["fips"], "name": c["name"], "state": c["state"],
            "median_income": c.get("median_income"),
            "unemp_2023": rates.get("2023"), "unemp_2018": rates.get("2018"),
            "unemp_2010": rates.get("2010"),
        })
    counties = pd.DataFrame(rows)
    print(f"{len(counties)} counties loaded")
    print(counties.groupby("state")[["unemp_2023","median_income"]].mean().round(2).to_string())


## County Unemployment Choropleth (Plotly)

In [ ]:
if "counties" in dir():
    COLORS = {"MD": "#4aa8d8", "VA": "#00FF41", "DC": "#D4AF37"}
    fig = px.choropleth(
        counties.dropna(subset=["unemp_2023"]),
        geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
        locations="fips", color="unemp_2023",
        scope="usa", fitbounds="locations",
        color_continuous_scale=[[0,"#00FF41"],[0.5,"#D4AF37"],[1.0,"#DC143C"]],
        hover_name="name",
        hover_data={"state": True, "unemp_2023": ":.1f", "median_income": ":,"},
        title="County Unemployment Rate 2023 — DMV Region",
        labels={"unemp_2023": "Unemployment %"},
    )
    fig.update_layout(paper_bgcolor="#121212", font=dict(color="#e8e8e8"))
    fig.show()


## Highest Unemployment Counties

In [ ]:
if "counties" in dir():
    top = counties.dropna(subset=["unemp_2023"]).nlargest(15, "unemp_2023")
    print("Top 15 counties by 2023 unemployment rate:")
    print(top[["name","state","unemp_2023","unemp_2018","unemp_2010","median_income"]].to_string(index=False))


## Embedded Folium Map

In [ ]:
from pathlib import Path
map_path = ROOT / "maps" / "dmv_counties.html"
if map_path.exists():
    from IPython.display import IFrame
    display(IFrame(src=str(map_path), width="100%", height=500))
else:
    print("Map not yet generated. Run: python generate_county_map.py")
